# 01 - Data Preparation & Merging
**Enhancing E-Commerce Sales Forecasting Using Google Trends**

---

## Objectives
1. Load synthetic weekly sales data
2. Load Google Trends data
3. Merge datasets on Date column
4. Handle missing values
5. Create final dataset for modeling

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("Libraries loaded successfully!")

## Step 1: Load Synthetic Sales Data

In [ ]:
# Load sales data
sales_file = '../data/processed/synthetic_weekly_sales.csv'

if not Path(sales_file).exists():
    print("⚠️ Sales data not found!")
    print("Run: python src/create_synthetic_sales.py")
else:
    sales_df = pd.read_csv(sales_file)
    sales_df['Date'] = pd.to_datetime(sales_df['Date'])
    
    print(f"✓ Loaded sales data: {sales_df.shape}")
    print(f"\nDate range: {sales_df['Date'].min()} to {sales_df['Date'].max()}")
    print(f"\nCategories: {sales_df['Category'].unique()}")
    
    sales_df.head()

## Step 2: Load Google Trends Data

In [ ]:
# Find the latest Google Trends file
trends_dir = Path('../data/google_trends')
trends_files = list(trends_dir.glob('earbuds_trends_*_base.csv'))

if not trends_files:
    print("⚠️ Google Trends data not found!")
    print("Run: python src/fetch_google_trends.py")
else:
    # Use the most recent file
    latest_file = max(trends_files, key=lambda p: p.stat().st_mtime)
    
    trends_df = pd.read_csv(latest_file)
    trends_df['date'] = pd.to_datetime(trends_df['date'])
    
    print(f"✓ Loaded Google Trends: {trends_df.shape}")
    print(f"\nKeywords: {[col for col in trends_df.columns if col != 'date']}")
    print(f"\nDate range: {trends_df['date'].min()} to {trends_df['date'].max()}")
    
    trends_df.head()

## Step 3: Align Date Frequencies

Google Trends returns **weekly data**, our sales data is also **weekly**.
We need to ensure both are on the same weekly grid.

In [ ]:
# Resample both to weekly starting Sunday
trends_df = trends_df.set_index('date').resample('W-SUN').mean().reset_index()
trends_df.rename(columns={'date': 'Date'}, inplace=True)

print(f"✓ Trends resampled: {trends_df.shape}")
print(f"Date range: {trends_df['Date'].min()} to {trends_df['Date'].max()}")

## Step 4: Merge Sales + Google Trends

In [ ]:
# Merge on Date (inner join to keep only overlapping dates)
merged_df = pd.merge(
    sales_df,
    trends_df,
    on='Date',
    how='inner'
)

print(f"✓ Merged dataset: {merged_df.shape}")
print(f"\nColumns: {list(merged_df.columns)}")
print(f"\nMissing values:")
print(merged_df.isnull().sum())

merged_df.head(10)

## Step 5: Create Lagged Google Trends Features

**Key Insight:** Search interest typically **leads** sales by 1-4 weeks.
People search → then buy later.

In [ ]:
# Create lagged features for each trend keyword
trend_keywords = [col for col in trends_df.columns if col != 'Date']

for keyword in trend_keywords:
    for lag in [1, 2, 3, 4]:  # 1-4 weeks lag
        merged_df[f'{keyword}_lag{lag}'] = merged_df.groupby('Category')[keyword].shift(lag)

print(f"✓ Created lagged features")
print(f"Total columns now: {len(merged_df.columns)}")

# Drop rows with NaN created by lagging
merged_df = merged_df.dropna()
print(f"✓ After removing NaN: {merged_df.shape}")

## Step 6: Feature Engineering - Additional Features

In [ ]:
# Time-based features
merged_df['Week_of_Year'] = merged_df['Date'].dt.isocalendar().week
merged_df['Month'] = merged_df['Date'].dt.month
merged_df['Quarter'] = merged_df['Date'].dt.quarter
merged_df['Is_Festive_Season'] = merged_df['Month'].isin([10, 11, 12]).astype(int)  # Oct-Dec

# Rolling averages of sales (2-week, 4-week)
merged_df['Units_Sold_MA2'] = merged_df.groupby('Category')['Units_Sold'].transform(
    lambda x: x.rolling(window=2, min_periods=1).mean()
)
merged_df['Units_Sold_MA4'] = merged_df.groupby('Category')['Units_Sold'].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean()
)

print("✓ Feature engineering complete")
print(f"\nFinal dataset shape: {merged_df.shape}")
merged_df.head()

## Step 7: Save Merged Dataset

In [ ]:
output_file = '../data/processed/merged_sales_trends.csv'
merged_df.to_csv(output_file, index=False)

print(f"✓ Saved to: {output_file}")
print(f"\nDataset ready for EDA (Notebook 02)!")

## Quick Preview: Sales vs. Google Trends

In [ ]:
# Select one category for visualization
category = merged_df['Category'].unique()[0]
cat_data = merged_df[merged_df['Category'] == category].copy()

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot sales
ax1.set_xlabel('Date')
ax1.set_ylabel('Units Sold', color='tab:blue')
ax1.plot(cat_data['Date'], cat_data['Units_Sold'], color='tab:blue', label='Units Sold', linewidth=2)
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(alpha=0.3)

# Plot Google Trends on secondary axis
ax2 = ax1.twinx()
ax2.set_ylabel('Google Trends Index', color='tab:red')
ax2.plot(cat_data['Date'], cat_data['earbuds india'], color='tab:red', label='Google Trends', linewidth=2, alpha=0.7)
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title(f'Sales vs. Google Trends: {category}', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()

print("\n📊 Notice: Google Trends peaks often PRECEDE sales peaks!")
print("This validates using lagged features for forecasting.")

---

## ✅ Checklist

- [x] Sales data loaded  
- [x] Google Trends data loaded  
- [x] Datasets merged on Date  
- [x] Lagged features created (1-4 weeks)  
- [x] Time-based features added  
- [x] Final dataset saved  

**Next:** Open `02_eda.ipynb` for Exploratory Data Analysis

---